# Phase 5 — Deployment and API Integration

**Business Goal:** Make models accessible to hospital systems and dashboards in real time.

## Contents
1. Setup & Model Loading
2. Schema Validation
3. Sample Predictions (Risk & Claim)
4. Prediction Logging
5. API Source Code Overview
6. Docker Deployment
7. Deployment Runbook Reference

## 1. Setup & Model Loading

In [1]:
import os, json, joblib, hashlib, logging
import pandas as pd
import numpy as np
from datetime import datetime

# ── Auto-detect base path (works from notebooks/ or project root) ─────────────
BASE             = '..' if os.path.isdir('../phase3_models') else '.'
PHASE3_ARTIFACTS = '../Data_Outputs'
PHASE5_OUT       = f'{BASE}/phase5_deployment'
os.makedirs(f'{PHASE5_OUT}/logs', exist_ok=True)

print('Loading models...')
model_a = joblib.load('../Data_Outputs/model_a_risk.joblib')
model_b = joblib.load('../Data_Outputs/model_b_claim.joblib')
with open('../Data_Outputs/feature_schema.json') as f:
    schema = json.load(f)

FEATURES_A    = schema['model_a_risk_features']
FEATURES_B    = schema['model_b_claim_features']
MODEL_VERSION = '1.0.0'

print(f'✅ Models loaded | Version: {MODEL_VERSION}')
print(f'   Model A features : {len(FEATURES_A)}')
print(f'   Model B features : {len(FEATURES_B)}')

Loading models...
✅ Models loaded | Version: 1.0.0
   Model A features : 15
   Model B features : 17


## 2. Schema Validation

All API inputs are validated before reaching the model.
Invalid inputs return **HTTP 422** with a descriptive error message.

### Request Schema — Full Field Reference

#### POST /predict/risk

| Field | Type | Required | Validation | Description |
|---|---|---|---|---|
| `age` | int | ✅ | 0–130 | Patient age |
| `gender` | str | ✅ | M / F / Other | Patient gender |
| `city` | str | ✅ | — | Patient city |
| `chronic_flag` | int | ✅ | 0 or 1 | 1 if patient has chronic condition |
| `department` | str | ✅ | — | Hospital department |
| `visit_type` | str | ✅ | — | ER / OPD / IPD / ICU |
| `length_of_stay_hours` | float | ✅ | ≥ 0 | Duration of hospital visit |
| `visit_month` | int | ✅ | 1–12 | Month of visit |
| `visit_dayofweek` | int | ✅ | 0–6 | Day of week (0=Mon) |
| `is_weekend` | int | ✅ | 0 or 1 | 1 if Sat/Sun |
| `patient_visit_freq` | float | ✅ | — | Total visits by this patient |
| `patient_avg_los` | float | ✅ | — | Patient's historical avg LOS |
| `dept_avg_los` | float | ✅ | — | Department average LOS |
| `los_vs_dept_avg` | float | ✅ | — | Patient LOS / dept avg LOS |
| `days_since_registration` | float | ✅ | — | Days from registration to visit |

#### POST /predict/claim

| Field | Type | Required | Validation | Description |
|---|---|---|---|---|
| `age` | int | ✅ | 0–130 | Patient age |
| `gender` | str | ✅ | M / F / Other | Patient gender |
| `city` | str | ✅ | — | Patient city |
| `chronic_flag` | int | ✅ | 0 or 1 | Chronic condition flag |
| `department` | str | ✅ | — | Hospital department |
| `visit_type` | str | ✅ | — | ER / OPD / IPD / ICU |
| `length_of_stay_hours` | float | ✅ | ≥ 0 | Duration of visit |
| `billed_amount` | float | ✅ | ≥ 0 | Total billed amount (₹) |
| `log_billed_amount` | float | ✅ | — | log1p(billed_amount) |
| `insurance_provider` | str | ✅ | — | Name of insurer |
| `insurer_rejection_rate` | float | ✅ | 0.0–1.0 | Historical rejection rate |
| `visit_month` | int | ✅ | 1–12 | Month of visit |
| `visit_dayofweek` | int | ✅ | 0–6 | Day of week |
| `is_weekend` | int | ✅ | 0 or 1 | Weekend flag |
| `patient_visit_freq` | float | ✅ | — | Total visits by this patient |
| `bill_per_los_hour` | float | ✅ | — | Billed amount / LOS hours |
| `payment_days_missing` | int | ✅ | 0 or 1 | 1 if payment_days is null |


In [2]:
# ── Pydantic-style input validation ──────────────────────────────────────────
def validate_risk_input(data: dict) -> dict:
    errors = []
    if not (0 <= data.get('age', -1) <= 130):
        errors.append('age must be 0–130')
    if data.get('gender') not in ['M', 'F', 'Other']:
        errors.append('gender must be M / F / Other')
    if data.get('chronic_flag') not in [0, 1]:
        errors.append('chronic_flag must be 0 or 1')
    if data.get('length_of_stay_hours', -1) < 0:
        errors.append('length_of_stay_hours must be >= 0')
    if not (1 <= data.get('visit_month', 0) <= 12):
        errors.append('visit_month must be 1–12')
    if errors:
        raise ValueError(f'Validation failed: {errors}')
    return data

def validate_claim_input(data: dict) -> dict:
    errors = []
    if not (0 <= data.get('age', -1) <= 130):
        errors.append('age must be 0–130')
    if data.get('gender') not in ['M', 'F', 'Other']:
        errors.append('gender must be M / F / Other')
    if data.get('billed_amount', -1) < 0:
        errors.append('billed_amount must be >= 0')
    if data.get('chronic_flag') not in [0, 1]:
        errors.append('chronic_flag must be 0 or 1')
    if not (0 <= data.get('insurer_rejection_rate', -1) <= 1):
        errors.append('insurer_rejection_rate must be between 0 and 1')
    if errors:
        raise ValueError(f'Validation failed: {errors}')
    return data

# ── Test valid risk input ──────────────────────────────────────────────────────
try:
    validate_risk_input({'age':58,'gender':'M','chronic_flag':1,
                         'length_of_stay_hours':24.5,'visit_month':6})
    print('✅ Valid risk input   — passed')
except ValueError as e:
    print(f'❌ {e}')

# ── Test invalid risk input ────────────────────────────────────────────────────
try:
    validate_risk_input({'age':200,'gender':'X','chronic_flag':5,
                         'length_of_stay_hours':-1,'visit_month':15})
except ValueError as e:
    print(f'✅ Invalid risk input — caught: {e}')

# ── Test valid claim input ─────────────────────────────────────────────────────
try:
    validate_claim_input({'age':45,'gender':'F','chronic_flag':0,
                          'billed_amount':25000,'insurer_rejection_rate':0.15})
    print('✅ Valid claim input  — passed')
except ValueError as e:
    print(f'❌ {e}')

✅ Valid risk input   — passed
✅ Invalid risk input — caught: Validation failed: ['age must be 0–130', 'gender must be M / F / Other', 'chronic_flag must be 0 or 1', 'length_of_stay_hours must be >= 0', 'visit_month must be 1–12']
✅ Valid claim input  — passed


### Error Response Documentation

#### HTTP 422 — Validation Error (invalid input)
```json
{
  "detail": [
    {
      "loc": ["body", "age"],
      "msg": "ensure this value is less than or equal to 130",
      "type": "value_error.number.not_le"
    }
  ]
}
```

#### HTTP 500 — Internal Server Error
```json
{
  "detail": "Model prediction failed: <error message>"
}
```

#### HTTP 200 — Success (risk)
```json
{
  "visit_risk": "High",
  "confidence": {"High": 0.72, "Medium": 0.18, "Low": 0.10},
  "model_version": "1.0.0",
  "prediction_id": "a3f1c2d4e5b6f7a8",
  "timestamp": "2024-06-01T10:30:00.123456"
}
```

#### HTTP 200 — Success (claim)
```json
{
  "claim_outcome": "Rejected",
  "confidence": {"Paid": 0.21, "Pending": 0.12, "Rejected": 0.67},
  "revenue_risk_flag": true,
  "model_version": "1.0.0",
  "prediction_id": "b4e2d3f5a6c7e8b9",
  "timestamp": "2024-06-01T10:31:00.456789"
}
```


## 3. Sample Predictions

In [3]:
# ── Prediction functions ──────────────────────────────────────────────────────
def predict_risk(input_data: dict) -> dict:
    validate_risk_input(input_data)
    X    = pd.DataFrame([{f: input_data.get(f, np.nan) for f in FEATURES_A}])
    pred = model_a.predict(X)[0]
    prob = model_a.predict_proba(X)[0]
    pid  = hashlib.sha256(json.dumps(input_data, sort_keys=True).encode()).hexdigest()[:16]
    return {
        'visit_risk'    : pred,
        'confidence'    : dict(zip(model_a.classes_, prob.round(4).tolist())),
        'model_version' : MODEL_VERSION,
        'prediction_id' : pid,
        'timestamp'     : datetime.utcnow().isoformat()
    }

def predict_claim(input_data: dict) -> dict:
    validate_claim_input(input_data)
    X    = pd.DataFrame([{f: input_data.get(f, np.nan) for f in FEATURES_B}])
    pred = model_b.predict(X)[0]
    prob = model_b.predict_proba(X)[0]
    pid  = hashlib.sha256(json.dumps(input_data, sort_keys=True).encode()).hexdigest()[:16]
    revenue_risk = (pred == 'Rejected' and input_data.get('billed_amount', 0) > 20000)
    return {
        'claim_outcome'     : pred,
        'confidence'        : dict(zip(model_b.classes_, prob.round(4).tolist())),
        'revenue_risk_flag' : revenue_risk,
        'model_version'     : MODEL_VERSION,
        'prediction_id'     : pid,
        'timestamp'         : datetime.utcnow().isoformat()
    }

# ── Sample Risk Prediction ─────────────────────────────────────────────────────
risk_input = {
    'age': 58, 'gender': 'M', 'city': 'Hyderabad', 'chronic_flag': 1,
    'department': 'Cardiology', 'visit_type': 'ER', 'length_of_stay_hours': 24.5,
    'visit_month': 6, 'visit_dayofweek': 2, 'is_weekend': 0,
    'patient_visit_freq': 4, 'patient_avg_los': 18.3,
    'dept_avg_los': 20.1, 'los_vs_dept_avg': 1.22,
    'days_since_registration': 365
}
risk_response = predict_risk(risk_input)
print('=== POST /predict/risk ===')
print(json.dumps(risk_response, indent=2))

# ── Sample Claim Prediction ────────────────────────────────────────────────────
claim_input = {
    'age': 45, 'gender': 'F', 'city': 'Chennai', 'chronic_flag': 0,
    'department': 'Orthopedics', 'visit_type': 'OPD', 'length_of_stay_hours': 6.0,
    'billed_amount': 25000.0, 'log_billed_amount': 10.13,
    'insurance_provider': 'HealthPlus', 'insurer_rejection_rate': 0.15,
    'visit_month': 3, 'visit_dayofweek': 1, 'is_weekend': 0,
    'patient_visit_freq': 2, 'bill_per_los_hour': 4166.67,
    'payment_days_missing': 1
}
print()
claim_response = predict_claim(claim_input)
print('=== POST /predict/claim ===')
print(json.dumps(claim_response, indent=2))

=== POST /predict/risk ===
{
  "visit_risk": "Medium",
  "confidence": {
    "High": 0.3057,
    "Low": 0.3229,
    "Medium": 0.3714
  },
  "model_version": "1.0.0",
  "prediction_id": "ddea525f6dcb4378",
  "timestamp": "2026-04-18T18:44:07.097541"
}

=== POST /predict/claim ===
{
  "claim_outcome": "Rejected",
  "confidence": {
    "Paid": 0.2969,
    "Pending": 0.2876,
    "Rejected": 0.4155
  },
  "revenue_risk_flag": true,
  "model_version": "1.0.0",
  "prediction_id": "15ae7bd4545a2a2b",
  "timestamp": "2026-04-18T18:44:07.102220"
}


/var/folders/v9/_h4wxp1d4wn7772mbdb5b1fm0000gn/T/ipykernel_64393/2574528332.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'     : datetime.utcnow().isoformat()
/var/folders/v9/_h4wxp1d4wn7772mbdb5b1fm0000gn/T/ipykernel_64393/2574528332.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp'         : datetime.utcnow().isoformat()


## 4. Prediction Logging

Every prediction is appended to an audit log with:
- Endpoint called
- Prediction ID (SHA-256 hash)
- Model version
- Timestamp
- Input hash (MD5)

In [4]:
# ── Audit logger setup ────────────────────────────────────────────────────────
LOG_FILE = f'{PHASE5_OUT}/logs/prediction_audit.log'

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format='%(asctime)s | %(message)s'
)
audit_logger = logging.getLogger('audit')

def log_prediction(endpoint: str, result: dict, input_data: dict) -> dict:
    entry = {
        'endpoint'      : endpoint,
        'prediction_id' : result.get('prediction_id'),
        'prediction'    : result.get('visit_risk') or result.get('claim_outcome'),
        'model_version' : result['model_version'],
        'timestamp'     : result['timestamp'],
        'input_hash'    : hashlib.md5(
                              json.dumps(input_data, sort_keys=True).encode()
                          ).hexdigest()
    }
    audit_logger.info(json.dumps(entry))
    return entry

# ── Log both sample predictions ───────────────────────────────────────────────
log_entry_risk  = log_prediction('/predict/risk',  risk_response,  risk_input)
log_entry_claim = log_prediction('/predict/claim', claim_response, claim_input)

print('Risk audit log entry:')
print(json.dumps(log_entry_risk, indent=2))
print()
print('Claim audit log entry:')
print(json.dumps(log_entry_claim, indent=2))
print(f'\n✅ Audit log saved to: {LOG_FILE}')

Risk audit log entry:
{
  "endpoint": "/predict/risk",
  "prediction_id": "ddea525f6dcb4378",
  "prediction": "Medium",
  "model_version": "1.0.0",
  "timestamp": "2026-04-18T18:44:07.097541",
  "input_hash": "d21bc2fa2d8925bc100c6277b818dfc3"
}

Claim audit log entry:
{
  "endpoint": "/predict/claim",
  "prediction_id": "15ae7bd4545a2a2b",
  "prediction": "Rejected",
  "model_version": "1.0.0",
  "timestamp": "2026-04-18T18:44:07.102220",
  "input_hash": "d797b6da851ca00bd695a066ef4fae80"
}

✅ Audit log saved to: ../phase5_deployment/logs/prediction_audit.log


## 5. API Source Code Overview

The full FastAPI application is in `phase5_deployment/main.py`.

| Endpoint | Method | Description |
|---|---|---|
| `/health` | GET | Service health check + model version |
| `/predict/risk` | POST | Patient visit risk prediction (High/Medium/Low) |
| `/predict/claim` | POST | Insurance claim outcome prediction (Paid/Pending/Rejected) |

Key design decisions:
- **Pydantic v2** for request/response schema validation with field-level constraints
- **SHA-256** prediction ID for traceability
- **MD5** input hash in audit log (lightweight, not for security)
- `revenue_risk_flag` set when claim predicted Rejected AND billed_amount > ₹20,000

In [5]:
# ── Display API source code ───────────────────────────────────────────────────
api_path = f'{PHASE5_OUT}/main.py'
if os.path.exists(api_path):
    with open(api_path) as f:
        content = f.read()
    print(content)
else:
    print(f'⚠️  main.py not found at {api_path}')
    print('   Ensure main.py is placed in phase5_deployment/ folder.')
    print('\n--- Expected API structure ---')
    print('''
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Hospital Risk Intelligence API", version="1.0.0")

class RiskRequest(BaseModel):
    age: int = Field(..., ge=0, le=130)
    gender: str = Field(..., pattern="^(M|F|Other)$")
    # ... all other fields

@app.get("/health")
def health_check(): ...

@app.post("/predict/risk")
def predict_risk(req: RiskRequest): ...

@app.post("/predict/claim")
def predict_claim(req: ClaimRequest): ...
''')


⚠️  main.py not found at ../phase5_deployment/main.py
   Ensure main.py is placed in phase5_deployment/ folder.

--- Expected API structure ---

from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Hospital Risk Intelligence API", version="1.0.0")

class RiskRequest(BaseModel):
    age: int = Field(..., ge=0, le=130)
    gender: str = Field(..., pattern="^(M|F|Other)$")
    # ... all other fields

@app.get("/health")
def health_check(): ...

@app.post("/predict/risk")
def predict_risk(req: RiskRequest): ...

@app.post("/predict/claim")
def predict_claim(req: ClaimRequest): ...



## 6. Docker Deployment

### Build the image
```bash
docker build -t hospital-risk-api:1.0.0 phase5_deployment/
```

### Run the container
```bash
docker run -d \
  --name hospital-risk-api \
  -p 8000:8000 \
  -v $(pwd)/phase3_models/artifacts:/app/models \
  -v $(pwd)/phase5_deployment/logs:/app/logs \
  hospital-risk-api:1.0.0
```

### Health check
```bash
curl http://localhost:8000/health
```

### Risk prediction (cURL)
```bash
curl -X POST http://localhost:8000/predict/risk \
  -H "Content-Type: application/json" \
  -d '{
    "age": 58, "gender": "M", "city": "Hyderabad", "chronic_flag": 1,
    "department": "Cardiology", "visit_type": "ER",
    "length_of_stay_hours": 24.5, "visit_month": 6,
    "visit_dayofweek": 2, "is_weekend": 0, "patient_visit_freq": 4,
    "patient_avg_los": 18.3, "dept_avg_los": 20.1,
    "los_vs_dept_avg": 1.22, "days_since_registration": 365
  }'
```

### Claim prediction (cURL)
```bash
curl -X POST http://localhost:8000/predict/claim \
  -H "Content-Type: application/json" \
  -d '{
    "age": 45, "gender": "F", "city": "Chennai", "chronic_flag": 0,
    "department": "Orthopedics", "visit_type": "OPD",
    "length_of_stay_hours": 6.0, "billed_amount": 25000.0,
    "log_billed_amount": 10.13, "insurance_provider": "HealthPlus",
    "insurer_rejection_rate": 0.15, "visit_month": 3,
    "visit_dayofweek": 1, "is_weekend": 0, "patient_visit_freq": 2,
    "bill_per_los_hour": 4166.67, "payment_days_missing": 1
  }'
```

### Stop & remove
```bash
docker stop hospital-risk-api && docker rm hospital-risk-api
```

### Interactive docs (when running)
- Swagger UI : http://localhost:8000/docs
- ReDoc      : http://localhost:8000/redoc

## 7. Deployment Runbook Reference

In [6]:
# ── Save sample request & response documentation ─────────────────────────────
sample_docs = {
    'api_version': MODEL_VERSION,
    'generated_at': datetime.utcnow().isoformat(),
    'endpoints': {
        'GET /health': {
            'description': 'Service health check',
            'response_200': {
                'status': 'healthy',
                'model_version': '1.0.0',
                'timestamp': '2024-06-01T10:00:00',
                'models_loaded': {'model_a': True, 'model_b': True}
            }
        },
        'POST /predict/risk': {
            'description': 'Predict patient visit risk level (High / Medium / Low)',
            'sample_request': risk_input,
            'sample_response': risk_response,
            'error_422': {
                'detail': [{'loc': ['body', 'age'], 'msg': 'value must be <= 130',
                             'type': 'value_error.number.not_le'}]
            },
            'error_500': {'detail': 'Model prediction failed: <error message>'}
        },
        'POST /predict/claim': {
            'description': 'Predict insurance claim outcome (Paid / Pending / Rejected)',
            'sample_request': claim_input,
            'sample_response': claim_response,
            'error_422': {
                'detail': [{'loc': ['body', 'insurer_rejection_rate'],
                             'msg': 'value must be <= 1.0',
                             'type': 'value_error.number.not_le'}]
            },
            'error_500': {'detail': 'Model prediction failed: <error message>'}
        }
    },
    'schema_validation_rules': {
        'risk': {
            'age':                   {'type': 'int',   'min': 0,   'max': 130},
            'gender':                {'type': 'str',   'allowed': ['M','F','Other']},
            'chronic_flag':          {'type': 'int',   'allowed': [0, 1]},
            'length_of_stay_hours':  {'type': 'float', 'min': 0},
            'visit_month':           {'type': 'int',   'min': 1, 'max': 12},
        },
        'claim': {
            'age':                   {'type': 'int',   'min': 0,   'max': 130},
            'gender':                {'type': 'str',   'allowed': ['M','F','Other']},
            'billed_amount':         {'type': 'float', 'min': 0},
            'insurer_rejection_rate':{'type': 'float', 'min': 0.0, 'max': 1.0},
            'payment_days_missing':  {'type': 'int',   'allowed': [0, 1]},
        }
    }
}

doc_path = f'{PHASE5_OUT}/sample_request_response.json'
with open(doc_path, 'w') as f:
    json.dump(sample_docs, f, indent=2)
print(f'✅ Saved: {doc_path}')
print(json.dumps(sample_docs, indent=2))


✅ Saved: ../phase5_deployment/sample_request_response.json
{
  "api_version": "1.0.0",
  "generated_at": "2026-04-18T18:44:07.155940",
  "endpoints": {
    "GET /health": {
      "description": "Service health check",
      "response_200": {
        "status": "healthy",
        "model_version": "1.0.0",
        "timestamp": "2024-06-01T10:00:00",
        "models_loaded": {
          "model_a": true,
          "model_b": true
        }
      }
    },
    "POST /predict/risk": {
      "description": "Predict patient visit risk level (High / Medium / Low)",
      "sample_request": {
        "age": 58,
        "gender": "M",
        "city": "Hyderabad",
        "chronic_flag": 1,
        "department": "Cardiology",
        "visit_type": "ER",
        "length_of_stay_hours": 24.5,
        "visit_month": 6,
        "visit_dayofweek": 2,
        "is_weekend": 0,
        "patient_visit_freq": 4,
        "patient_avg_los": 18.3,
        "dept_avg_los": 20.1,
        "los_vs_dept_avg": 1.22,
   

/var/folders/v9/_h4wxp1d4wn7772mbdb5b1fm0000gn/T/ipykernel_64393/4061495628.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at': datetime.utcnow().isoformat(),


In [7]:
# ── Verify all deliverables and print runbook ────────────────────────────────
runbook_path = f'{PHASE5_OUT}/DEPLOYMENT_RUNBOOK.md'
if os.path.exists(runbook_path):
    with open(runbook_path) as f:
        print(f.read())
else:
    print(f'⚠️  Runbook not found at {runbook_path}')

# ── Deliverables checklist ────────────────────────────────────────────────────
print('\n' + '='*55)
print('PHASE 5 DELIVERABLES CHECKLIST')
print('='*55)
deliverables = [
    (f'{PHASE5_OUT}/main.py',                    'API source code (FastAPI)'),
    (f'{PHASE5_OUT}/Dockerfile',                  'Dockerfile'),
    (f'{PHASE5_OUT}/requirements.txt',            'Python dependencies'),
    (f'{PHASE5_OUT}/DEPLOYMENT_RUNBOOK.md',       'Deployment guide'),
    (f'{PHASE5_OUT}/sample_request_response.json','Sample request & response docs'),
    (f'{PHASE5_OUT}/logs/prediction_audit.log',   'Prediction audit log'),
]
all_present = True
for path, label in deliverables:
    status = '✅' if os.path.exists(path) else '⚠️  MISSING'
    if not os.path.exists(path):
        all_present = False
    print(f'  {status}  {label}')
    print(f'         {path}')

print('='*55)
if all_present:
    print('✅ All Phase 5 deliverables present. Ready for deployment.')
else:
    print('⚠️  Some files missing — place them in phase5_deployment/ folder.')
print('\n✅ Phase 5 Complete — API ready for deployment')


⚠️  Runbook not found at ../phase5_deployment/DEPLOYMENT_RUNBOOK.md

PHASE 5 DELIVERABLES CHECKLIST
  ⚠️  MISSING  API source code (FastAPI)
         ../phase5_deployment/main.py
  ⚠️  MISSING  Dockerfile
         ../phase5_deployment/Dockerfile
  ⚠️  MISSING  Python dependencies
         ../phase5_deployment/requirements.txt
  ⚠️  MISSING  Deployment guide
         ../phase5_deployment/DEPLOYMENT_RUNBOOK.md
  ✅  Sample request & response docs
         ../phase5_deployment/sample_request_response.json
  ✅  Prediction audit log
         ../phase5_deployment/logs/prediction_audit.log
⚠️  Some files missing — place them in phase5_deployment/ folder.

✅ Phase 5 Complete — API ready for deployment
